# Assemble datasets from simulations
Combine data from simulations of different network architectures

In [1]:
import numpy as np
import pandas as pd
import os
import pickle
from tqdm import tqdm
from joblib import Parallel, delayed
import re

from stoch_sim_model import *

In [2]:
# Set parameters
sim_kind = 'agent'
reg_model = ''
runs = '-1-'
comment = "acute_all-sparse-reg"

d = '/gscratch/scrubbed/oukogu/infoimmune/sim_output/no_cell_var/'

sim_sum_list = []
parameters_nets = []
prim_diff_bias_list = []
#sec_diff_bias_list = []
cell_series_list = []
# lineage_diff_nets = []

In [3]:
# Figure out which jobs didn't run:
d_rerun = '/gscratch/scrubbed/oukogu/infoimmune/sim_output/no_cell_var/raw/'
run_list = [int(re.search('sim_batch_(.*?)\.', f).group(1)) for f in os.listdir(d_rerun) if 'sim_batch' in f and comment in f and runs in f]
out = [str(x) for x in [k for k in np.arange(0, 589)] if x not in run_list]
print(len(out))
print(' '.join((out)))

0



In [4]:
num_cpu = 125
file_list = [f for f in os.listdir(os.path.join(d, "raw")) if runs in f and comment in f and 'sim_batch' in f]
num_files = len(file_list)

def import_dict_func(f,d):
    
    file_path = os.path.join(os.path.join(d, "raw"), f)
    with open(file_path, 'rb') as filename:  
        import_dict = pickle.load(filename)

    parameters = np.array(import_dict["parameters"])
    sim_sum = np.array(import_dict["summary_stats"])

    out = np.hstack((parameters, sim_sum))

    return out

# create dataframe of infection response statistics
var_names = np.concatenate((param_names_for_df, stat_names_for_df))
# mean_df = pd.DataFrame(np.vstack(Parallel(n_jobs = num_cpu, batch_size = max(int(num_files/num_cpu),1))(delayed(import_dict_func)(f = file_name, d = d) 
#                                                                                                         for file_name in file_list)), 
#                        columns = [i for i in var_names]).groupby(Na_reg + NE_reg + EM_reg + EE_reg + vir_vars, as_index=False).mean()
full_df = pd.DataFrame(np.vstack(Parallel(n_jobs = num_cpu, batch_size = max(int(num_files/num_cpu),1))(delayed(import_dict_func)(f = file_name, d = d) 
                                                                                                        for file_name in file_list)), 
                       columns = [i for i in var_names])

# # Save datasets
full_df.to_pickle(os.path.join(d, "raw", "stacked_full_data"+runs+"runs"+'-'+comment)+'.pkl')

In [5]:
with pd.option_context('display.max_columns', None):
    display(full_df)

,S_0,I_0,b_I,d_S,d_I,d_IE,K_I,b_H,d_H,K_H,N_0,max_Na,b_myc,d_myc,myc_thresh,t_bind,t_unbind,t_Na_div,t_E_div,t_M_div,t_E_die,t_cycle,psi_myc_I,psi_myc_HI,psi_myc_HE,L0_Na,psi_NE_I,psi_NE_HI,psi_NE_HE,L0_NE,psi_EM_I,psi_EM_HI,psi_EM_HE,L0_EM,psi_Edie_I,psi_Edie_HI,psi_Edie_HE,L0_Edie,p_load,T_max_pI,T_min_pI,harm_pI,harm_pS,max_pE,T_pE_max,T_pE_start,max_eM,T_pEcyteM,T_pE_end,frac_cM,int_pHE,int_pHI,min_pS
0,10000000.0,1000.0,1.000000e-07,0.01,0.1,12.0,10000.0,1.0,2.0,10000.0,100.0,4.0,144.0,49.906597,1.0,0.5,1.0,0.366667,0.333333,0.5,10.0,0.25,-1.5,2.0,1.5,1.50,-1.5,2.0,1.5,-3.00,-0.0,0.0,0.0,-27.0,1.5,-2.0,-1.5,0.00,6.872123e+06,13.10,0.00,1.301851e+07,446.047872,396.0,2.83,0.53,0.0,0.0,0.0,0.933535,199.559556,340620.700566,2.063764e+05
1,10000000.0,1000.0,1.000000e-07,0.01,0.1,12.0,10000.0,1.0,2.0,10000.0,100.0,4.0,144.0,49.906597,1.0,0.5,1.0,0.366667,0.333333,0.5,10.0,0.25,-1.5,2.0,1.5,1.50,-1.5,2.0,1.5,-3.00,-0.0,0.0,0.0,-27.0,1.5,-2.0,-1.5,0.75,6.872793e+06,13.07,0.00,1.302105e+07,99.852633,242.0,2.19,0.43,0.0,0.0,0.0,0.916149,41.455775,340653.577979,2.063554e+05
2,10000000.0,1000.0,1.000000e-07,0.01,0.1,12.0,10000.0,1.0,2.0,10000.0,100.0,4.0,144.0,49.906597,1.0,0.5,1.0,0.366667,0.333333,0.5,10.0,0.25,-1.5,2.0,1.5,1.50,-1.5,2.0,1.5,-3.00,-0.0,0.0,0.0,-27.0,1.5,-2.0,-1.5,1.50,6.872569e+06,13.07,0.00,1.302044e+07,206.110901,291.0,4.14,0.41,0.0,0.0,0.0,0.936782,107.976001,340642.401907,2.063626e+05
3,10000000.0,1000.0,1.000000e-07,0.01,0.1,12.0,10000.0,1.0,2.0,10000.0,100.0,4.0,144.0,49.906597,1.0,0.5,1.0,0.366667,0.333333,0.5,10.0,0.25,-1.5,2.0,1.5,1.50,-1.5,2.0,1.5,-3.00,-0.0,0.0,0.0,-27.0,1.5,-2.0,-1.5,2.25,6.872848e+06,13.06,0.00,1.302145e+07,65.243499,230.0,2.16,0.52,0.0,0.0,0.0,0.938272,26.886589,340656.125362,2.063538e+05
4,10000000.0,1000.0,1.000000e-07,0.01,0.1,12.0,10000.0,1.0,2.0,10000.0,100.0,4.0,144.0,49.906597,1.0,0.5,1.0,0.366667,0.333333,0.5,10.0,0.25,-1.5,2.0,1.5,1.50,-1.5,2.0,1.5,-3.00,-0.0,0.0,0.0,-27.0,1.5,-2.0,-1.5,3.00,6.872546e+06,13.08,0.00,1.302004e+07,231.482502,316.0,3.20,0.56,0.0,0.0,0.0,0.929412,103.090495,340641.428419,2.063632e+05
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
106288195,10000000.0,1000.0,1.000000e-07,0.01,1.0,12.0,10000000.0,1.0,2.0,100000.0,100.0,4.0,144.0,49.906597,1.0,0.5,1.0,0.366667,0.333333,0.5,10.0,0.25,0.5,-1.0,-1.0,-2.25,0.5,-1.0,-1.0,-0.75,0.0,-0.0,-0.0,-27.0,-0.5,1.0,1.0,1.50,1.000000e+03,0.00,0.02,1.020000e+03,0.000000,200.0,0.00,50.00,0.0,0.0,0.0,0.000000,0.000000,256.786870,9.999980e+06
106288196,10000000.0,1000.0,1.000000e-07,0.01,1.0,12.0,10000000.0,1.0,2.0,100000.0,100.0,4.0,144.0,49.906597,1.0,0.5,1.0,0.366667,0.333333,0.5,10.0,0.25,0.5,-1.0,-1.0,-2.25,0.5,-1.0,-1.0,-0.75,0.0,-0.0,-0.0,-27.0,-0.5,1.0,1.0,2.25,1.000000e+03,0.00,0.02,1.020000e+03,0.000000,200.0,0.00,50.00,0.0,0.0,0.0,0.000000,0.000000,256.786870,9.999980e+06
106288197,10000000.0,1000.0,1.000000e-07,0.01,1.0,12.0,10000000.0,1.0,2.0,100000.0,100.0,4.0,144.0,49.906597,1.0,0.5,1.0,0.366667,0.333333,0.5,10.0,0.25,0.5,-1.0,-1.0,-2.25,0.5,-1.0,-1.0,-0.75,0.0,-0.0,-0.0,-27.0,-0.5,1.0,1.0,3.00,1.000000e+03,0.00,0.02,1.020000e+03,0.000000,200.0,0.00,50.00,0.0,0.0,0.0,0.000000,0.000000,256.786870,9.999980e+06
106288198,10000000.0,1000.0,1.000000e-07,0.01,1.0,12.0,10000000.0,1.0,2.0,100000.0,100.0,4.0,144.0,49.906597,1.0,0.5,1.0,0.366667,0.333333,0.5,10.0,0.25,0.5,-1.0,-1.0,-2.25,0.5,-1.0,-1.0,0.00,0.0,-0.0,-0.0,-27.0,-0.5,1.0,1.0,-3.00,1.000000e+03,0.00,0.02,1.020000e+03,0.000000,200.0,0.00,50.00,0.0,0.0,0.0,0.000000,0.000000,256.786870,9.999980e+06


In [6]:
# Create additional variables
virs = np.unique(full_df[['I_0','d_I','K_I','b_I','K_H','N_0']].values, axis = 0)

full_df['antigenicity_over_harm'] = antigenicity_over_harm(full_df)
full_df['stim_pI'] = np.log(1 + (full_df['p_load']/full_df['K_I']))
full_df['stim_pHI'] = np.log(1 + (full_df['int_pHI']/full_df['K_H']))
full_df['stim_pHE'] = np.log(1 + (full_df['int_pHE']/full_df['K_H']))
full_df['scaled_min_pS'] = full_df['min_pS']/full_df['S_0']

# identify Biologically evidenced networks
keep_vars = ['harm_pI', 'harm_pS', 'frac_cM', 'max_pE',
             'T_pE_start', 'T_pE_max', 'T_pE_end',
             'stim_pI', 'stim_pHI', 'stim_pHE',
             'scaled_min_pS', 'antigenicity_over_harm']

In [7]:
# save data sets
full_infection_scenarios = []
mean_of_infection_scenarios = []
std_of_infection_scenarios = []
no_eff_data = [[] for i in np.arange(len(virs))]
b_S = d_S*S_0

for l, (I_0, d_I, K_I, b_I, K_H, N_0) in enumerate(tqdm(virs)):
    data = full_df.loc[(full_df["d_I"] == d_I)*(full_df["K_I"] == K_I)*(full_df["b_I"] == b_I)*(full_df["K_H"] == K_H)*(full_df["N_0"] == N_0)*(full_df["I_0"] == I_0), 
    ['b_I','d_I', 'K_I', 'I_0','S_0', 'N_0', 'd_S', 'K_H'] + Na_reg + NE_reg + EM_reg + EE_reg + keep_vars]

    # compute infection harm without T cell response
    no_eff_data[l] = lin_stoch_sim(N_0 = 0, I_0 = I_0, K_I = K_I, d_I = d_I, b_I = b_I,
                                   infection_model = "cancer" if b_I >= b_C else "acute")
    no_eff_stats = no_eff_data[l]["summary_stats"]

    data.loc[:,"harm_pI_noprotection"] = no_eff_stats[3]/S_0
    data.loc[:,"peff_clearance"] = (no_eff_stats[3] - data['harm_pI'].to_numpy())/S_0
    data.loc[:,"peff_toxicity"] = data['harm_pS'].to_numpy()/S_0
    data.loc[:,"peff_protection"] = (data['peff_clearance'] - data['peff_toxicity'])/data['harm_pI_noprotection']

    mean_of_infection_scenarios.append(data.groupby(Na_reg + NE_reg + EM_reg + EE_reg + vir_vars, as_index=False).mean())
    std_of_infection_scenarios.append(data.groupby(Na_reg + NE_reg + EM_reg + EE_reg + vir_vars, as_index=False).std())
    full_infection_scenarios.append(data)

# stack datasets
pd.concat(mean_of_infection_scenarios).to_pickle('/gscratch/scrubbed/oukogu/infoimmune/sim_output/no_cell_var/summary_stats/mean/processed_data'+runs+'runs'+'-'+comment+'.pkl')
pd.concat(std_of_infection_scenarios).to_pickle('/gscratch/scrubbed/oukogu/infoimmune/sim_output/no_cell_var/summary_stats/std/processed_data'+runs+'runs'+'-'+comment+'.pkl')
pd.concat(full_infection_scenarios).to_pickle('/gscratch/scrubbed/oukogu/infoimmune/sim_output/no_cell_var/summary_stats/processed_full_data'+runs+'runs'+'-'+comment+'.pkl')

with open('/gscratch/scrubbed/oukogu/infoimmune/sim_output/no_cell_var/summary_stats/mean/list_processed_data'+runs+'runs'+'-'+comment+'.pkl', 'wb') as f:
    pickle.dump(mean_of_infection_scenarios, f)

with open('/gscratch/scrubbed/oukogu/infoimmune/sim_output/no_cell_var/summary_stats/std/list_processed_data'+runs+'runs'+'-'+comment+'.pkl', 'wb') as f:
    pickle.dump(std_of_infection_scenarios, f)

with open('/gscratch/scrubbed/oukogu/infoimmune/sim_output/no_cell_var/summary_stats/list_processed_full_data'+runs+'runs'+'-'+comment+'.pkl', 'wb') as f:
    pickle.dump(full_infection_scenarios, f)

100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 200/200 [37:35<00:00, 11.28s/it]


In [8]:
# Clear memory
del full_df, mean_of_infection_scenarios, std_of_infection_scenarios, full_infection_scenarios